# 📘 PIEC Driver Development Tutorial & Test Template

This notebook serves as both a **tutorial** for understanding the PIEC driver folder structure and a **template** for creating test notebooks for new drivers.

---

## Table of Contents
1. [Driver Architecture Overview](#1-driver-architecture-overview)
2. [Folder Structure](#2-folder-structure)
3. [Class Hierarchy (3 Levels)](#3-class-hierarchy)
4. [Naming Conventions](#4-naming-conventions)
5. [Class Attributes & Parameter Restrictions](#5-class-attributes)
6. [Creating a New Driver — Step-by-Step](#6-creating-a-new-driver)
7. [Test Notebook Template](#7-test-notebook-template)
8. [Autodetect System](#8-autodetect-system)
9. [Checklist](#9-checklist)

---
## 1. Driver Architecture Overview <a id="1-driver-architecture-overview"></a>

PIEC uses a **3-level class hierarchy** to create instrument drivers:

```
Level 1: Base classes (communication protocol)
         ├── Instrument       (base for ALL instruments)
         ├── Scpi(Instrument)  (for SCPI-compliant instruments)
         └── Digilent          (for Digilent boards)

Level 2: Instrument-type base class (pseudo-code / interface)
         ├── awg.py, oscilloscope.py, dmm.py, lockin.py
         ├── sourcemeter.py, pulser.py, rf_source.py
         ├── daq.py, dc_callibrator.py, stepper_motor.py
         └── These define the INTERFACE — what methods every driver must implement

Level 3: Specific instrument driver
         ├── k_81150a.py, k_dsox3024a.py, srs830.py, etc.
         └── These IMPLEMENT the interface using the instrument's manual
```

**Key Insight:** Level 2 files define *what* a driver must do. Level 3 files define *how* a specific instrument does it.

---
## 2. Folder Structure <a id="2-folder-structure"></a>

Every instrument-type folder follows this pattern:

```
drivers/
├── instrument.py          # Level 1: Base class for ALL instruments
├── scpi.py                # Level 1: SCPI commands (reset, clear, error...)
├── autodetect.py          # Auto-detection logic
│
├── oscilloscope/          # One folder per instrument type
│   ├── __init__.py        # Required for Python package
│   ├── oscilloscope.py    # Level 2: Parent class (pseudo-code interface)
│   ├── k_dsox3024a.py     # Level 3: Keysight DSOX3024A driver
│   ├── lecroy_sda6020.py  # Level 3: LeCroy SDA6020 driver
│   └── scope_test.ipynb   # Test notebook for this instrument type
│
├── awg/
│   ├── __init__.py
│   ├── awg.py             # Level 2: Parent class
│   ├── k_81150a.py        # Level 3: Keysight 81150A driver
│   └── awg_test.ipynb
│
├── example/               # THIS folder — reference/tutorial
│   ├── example.py         # Example Level 2 parent class
│   ├── specific_example.py # Example Level 3 child driver
│   ├── DRIVER_DEVELOPMENT_GUIDE.md
│   ├── outline.md         # AI prompts & design notes
│   ├── readme.md
│   └── example_test.ipynb # THIS notebook
│   (NOTE: No __init__.py — intentionally excluded from package)
```

### Key Files in Each Folder

| File | Purpose |
|---|---|
| `__init__.py` | Makes the folder a Python package (required for `import`) |
| `<type>.py` | Level 2 parent class — the interface/pseudo-code |
| `<model>.py` | Level 3 specific driver — implements the interface |
| `<type>_test.ipynb` | Test notebook — verifies every method works |

---
## 3. Class Hierarchy (3 Levels) <a id="3-class-hierarchy"></a>

Here is the inheritance chain for a typical SCPI instrument:

In [ ]:
# This cell shows the class hierarchy — do not run, for illustration only

# === LEVEL 1: Base Classes ===
class Instrument:
    """Base for ALL instruments. Handles connection via PiecManager or VirtualRMInstrument."""
    def idn(self): ...

class Scpi(Instrument):
    """Standard SCPI commands available to all SCPI instruments."""
    def reset(self): ...       # *RST
    def clear(self): ...       # *CLS
    def error(self): ...       # *ESR?
    def self_test(self): ...   # *TST?
    def wait(self): ...        # *WAI
    def operation_complete(self): ...  # *OPC?
    def initialize(self): ...  # reset() + clear()

# === LEVEL 2: Instrument-Type Interface ===
class Awg(Instrument):   # or Oscilloscope, DMM, Lockin, Pulser, etc.
    """Defines the INTERFACE — what every AWG driver must implement."""
    channel = [1]
    waveform = ['SIN', 'SQU', 'RAMP']
    frequency = (None, None)
    
    def output(self, channel, on=True): ...
    def set_waveform(self, channel, waveform): ...
    def set_frequency(self, channel, frequency): ...
    def configure_waveform(self, channel, waveform, frequency=None, ...): ...

# === LEVEL 3: Specific Driver ===
class Keysight81150A(Awg, Scpi):
    """Specific driver — inherits BOTH the type interface AND SCPI commands."""
    AUTODETECT_ID = "81150A"
    channel = [1, 2]
    waveform = ['SIN', 'SQU', 'RAMP', 'PULS', 'NOIS', 'DC', 'USER']
    frequency = {'waveform': {'SIN': (1e-6, 240e6), 'SQU': (1e-6, 120e6)}}
    
    def output(self, channel, on=True):
        self.instrument.write(f":OUTP{channel} {'ON' if on else 'OFF'}")
    
    def set_frequency(self, channel, frequency):
        self.instrument.write(f":FREQ{channel} {frequency}")

### Multiple Inheritance Pattern

```python
class MyDriver(InstrumentType, Scpi):  # Level 3
    pass
```

- **First parent** = the instrument type (`Awg`, `Oscilloscope`, `DMM`, etc.)
- **Second parent** = the communication protocol (`Scpi`, or `Instrument` for non-SCPI)

This gives `MyDriver` access to:
- All the instrument-type interface methods (from parent class pseudo-code)
- All SCPI commands (`reset()`, `clear()`, `error()`, etc.) for free

---
## 4. Naming Conventions <a id="4-naming-conventions"></a>

### Method Names

| Prefix | Scope | Example |
|---|---|---|
| `set_` | **Single action** — writes one parameter | `set_frequency(ch, 1000)` |
| `configure_` | **Multiple actions** — calls several `set_` methods | `configure_waveform(ch, 'SIN', freq=1000, amp=1.0)` |
| `get_` | **Returns data** — queries/reads a value | `get_voltage()` → `float` |
| `read_` | **Returns data** — reads acquired data | `read_data()` → `DataFrame` |

### File Names
- Parent class: `<instrument_type>.py` → `awg.py`, `oscilloscope.py`
- Specific driver: `<manufacturer_prefix>_<model>.py` → `k_81150a.py`, `lecroy_sda6020.py`
- Test notebook: `<type>_test.ipynb` → `awg_test.ipynb`, `scope_test.ipynb`

### Constructor Rules
- **DO NOT** write `__init__` if it only calls `super().__init__()` — the base class handles it
- **IF** custom init is needed (hardware queries on connect), use `*args, **kwargs`:

```python
def __init__(self, *args, **kwargs):
    super().__init__(*args, **kwargs)
    # Custom post-connection queries here
```

---
## 5. Class Attributes & Parameter Restrictions <a id="5-class-attributes"></a>

Class attributes define what values each argument can take. The attribute name **MUST** match the argument name exactly.

### Three Formats

In [ ]:
# Format 1: LIST — for discrete sets of allowed values
channel = [1, 2]
waveform = ['SIN', 'SQU', 'RAMP', 'PULS']
trigger_source = ['IMM', 'INT', 'EXT', 'MAN']

# Format 2: TUPLE — for continuous ranges (min, max)
amplitude = (0.01, 10.0)   # Vpp
offset = (-5.0, 5.0)       # Volts
duty_cycle = (0.0, 100.0)  # percent

# Format 3: DICTIONARY — when valid range depends on another argument
# Key = argument name it depends on
# Value = dict mapping each option to its range
frequency = {
    'waveform': {
        'SIN': (1e-6, 240e6),    # sine goes up to 240 MHz
        'SQU': (1e-6, 120e6),    # square up to 120 MHz  
        'RAMP': (1e-6, 5e6),     # ramp up to 5 MHz
        'DC': None               # DC has no frequency
    }
}

# Special: None = unknown/not applicable — skipped by autocheck
rise_time = None

### Important Rules
1. Attribute name = argument name (e.g., `frequency` attribute validates `frequency` argument)
2. Parent class defines the minimum vocabulary — child class can **add** but never **remove**
3. If the parent says `channel = [1]` and your instrument has 2 channels: `channel = [1, 2]`
4. If a parent attribute is not applicable, set to `None` (autocheck skips it)
5. If a manual command uses different terms than the parent, **translate** in the method body:

```python
# Parent says waveform='SIN', but instrument expects 'SINE'
def set_waveform(self, channel, waveform):
    waveform_map = {'SIN': 'SINE', 'SQU': 'SQUARE', 'RAMP': 'RAMP'}
    self.instrument.write(f":FUNC {waveform_map[waveform]}")
```

---
## 6. Creating a New Driver — Step-by-Step <a id="6-creating-a-new-driver"></a>

### Step 1: Identify the instrument type
Which folder does it belong in? (awg, oscilloscope, dmm, lockin, pulser, etc.)

### Step 2: Create the driver file
```
drivers/<type>/<manufacturer_prefix>_<model>.py
```

### Step 3: Set up the class

In [ ]:
# Example: Creating a new AWG driver for the Agilent 33220A

# --- File: drivers/awg/agilent_33220a.py ---
from .awg import Awg
from ..scpi import Scpi

class Agilent33220a(Awg, Scpi):
    """Driver for the Agilent 33220A Arbitrary Waveform Generator."""
    
    # AUTODETECT_ID: unique substring from *IDN? response
    AUTODETECT_ID = "33220A"
    
    # Class attributes (from manual)
    channel = [1]                              # Single channel instrument
    waveform = ['SIN', 'SQU', 'RAMP', 'PULS', 'NOIS', 'DC', 'USER']
    frequency = (1e-6, 20e6)                   # 1 µHz to 20 MHz
    amplitude = (0.01, 10.0)                   # 10 mVpp to 10 Vpp (50Ω)
    offset = (-5.0, 5.0)                       # ±5 V
    
    # Implement every parent class method:
    def output(self, channel, on=True):
        """Turns the output on or off."""
        state = 'ON' if on else 'OFF'
        self.instrument.write(f"OUTP {state}")
    
    def set_waveform(self, channel, waveform):
        """Sets the waveform shape."""
        self.instrument.write(f"FUNC {waveform}")
    
    def set_frequency(self, channel, frequency):
        """Sets the frequency in Hz."""
        self.instrument.write(f"FREQ {frequency}")
    
    # ... override ALL remaining parent methods ...

### Step 4: Register for autodetect
The `AUTODETECT_ID` attribute is all you need. The autodetect system scans all `.py` files in the folder and matches the `*IDN?` response.

### Step 5: Update `__init__.py`
Add your class to the folder's `__init__.py` so it's importable.

### Step 6: Create or update the test notebook
Follow the test template below.

---
## 7. Test Notebook Template <a id="7-test-notebook-template"></a>

Every test notebook should follow this structure. Copy these cells as a starting point for your own `<type>_test.ipynb`.

### Section 0: Setup & Connection
Connect to the instrument using autodetect or manual address.

In [ ]:
# ====== TEMPLATE: SETUP ======
from piec.drivers.autodetect import autodetect

# Option A: Auto-detect by instrument type
instrument = autodetect("<instrument_type>", verbose=True)  # e.g. "awg", "oscilloscope"

# Option B: Manual connection
# from piec.drivers.<type>.<driver> import <DriverClass>
# instrument = <DriverClass>("<VISA_ADDRESS>")  # e.g. "GPIB0::15::INSTR"

### Section 1: Instrument & SCPI Tests
Always start with identification and standard SCPI commands.

In [ ]:
# ====== TEMPLATE: IDN ======
idn_response = instrument.idn()
print("IDN:", idn_response)
# Expected: "Manufacturer, Model, Serial, Firmware"

In [ ]:
# ====== TEMPLATE: SCPI SUITE ======
instrument.reset()                    # *RST — back to factory defaults
print("Reset: OK")

instrument.clear()                    # *CLS — clear status registers
print("Clear: OK")

print("Error:", instrument.error())   # *ESR? — expect 0
print("Self-test:", instrument.self_test())  # *TST? — expect 0
print("OPC:", instrument.operation_complete())  # *OPC? — expect 1

instrument.initialize()               # reset + clear
print("Initialize: OK")

### Section 2+: Type-Specific Tests
Test each method from the parent class, **in order of the class definition**.

Rules for each test cell:
- One clear action per cell
- Markdown cell **before or after** with **Expected Result** and **✅ PASS criteria**
- Use concrete, instrument-appropriate values
- For each `set_` method, follow with a verification (visual on instrument or query)
- For `configure_` methods, verify all sub-parameters changed

In [ ]:
# ====== TEMPLATE: SET METHOD TEST ======
# instrument.set_<parameter>(channel, value)
# print("<Parameter> set to <value>.")
#
# Expected Result: Describe what the technician should see on the instrument.
# ✅ PASS if <condition>

In [ ]:
# ====== TEMPLATE: CONFIGURE METHOD TEST ======
# instrument.configure_<module>(
#     channel=1,
#     param1=value1,
#     param2=value2,
#     param3=value3
# )
# print("Configured: param1=value1, param2=value2, param3=value3.")
#
# Expected Result: ALL sub-parameters should match.
# ✅ PASS if all match

In [ ]:
# ====== TEMPLATE: OUTPUT/MEASUREMENT TEST ======
# instrument.output(channel=1, on=True)
# result = instrument.quick_read()  # or get_voltage(), get_data(), etc.
# print(f"Result: {result}")
# instrument.output(channel=1, on=False)
#
# Expected Result: Describe the expected measurement value.
# ✅ PASS if <condition>

### Final Section: Cleanup & Summary

In [ ]:
# ====== TEMPLATE: CLEANUP ======
# instrument.output(channel=1, on=False)  # If applicable
instrument.reset()
print("Instrument reset to factory defaults.")

```
## Test Summary

| # | Method(s) Tested | Pass/Fail |
|---|------------------|-----------|
| 0 | __init__ / autodetect | |
| 1 | idn() | |
| 2 | reset(), clear(), error(), self_test(), operation_complete(), initialize() | |
| 3 | set_<param1>() | |
| 4 | set_<param2>() | |
| 5 | configure_<module>() | |
| 6 | output() | |
| 7 | quick_read() / get_data() | |
| 8 | reset() (cleanup) | |

**Technician Signature:** _______________  
**Date:** _______________  
**Instrument Serial #:** _______________
```

---
## 8. Autodetect System <a id="8-autodetect-system"></a>

The autodetect system works as follows:

1. `autodetect("awg")` scans the `drivers/awg/` folder
2. It finds all `.py` files and looks for classes with an `AUTODETECT_ID` attribute
3. It queries all connected VISA instruments with `*IDN?`
4. If an `*IDN?` response contains the `AUTODETECT_ID` string, it instantiates that class

### AUTODETECT_ID Examples

```python
# Single model
AUTODETECT_ID = "81150A"

# Multiple models handled by one driver
AUTODETECT_ID = ["81150A", "33522B", "33622A"]

# Unknown / not in manual
AUTODETECT_ID = None  # Autodetect will skip this driver
```

### Testing Autodetect

In [ ]:
# Test if autodetect can find your instrument
from piec.drivers.autodetect import autodetect

# This will print discovered instruments and their matched drivers
# instrument = autodetect("<type>", verbose=True)

---
## 9. Checklist for New Driver Development <a id="9-checklist"></a>

Use this checklist when creating a new driver:

- [ ] **Folder placement:** Driver is in the correct `drivers/<type>/` folder
- [ ] **Inheritance:** Class inherits from `(<Type>, Scpi)` or `(<Type>, Instrument)`
- [ ] **AUTODETECT_ID:** Set to unique model string from `*IDN?` response
- [ ] **Class attributes:** All argument restrictions defined (list, tuple, or dict)
- [ ] **Attribute names:** Match argument names exactly
- [ ] **All parent methods:** Every parent class method is overridden
- [ ] **set_ methods:** Each does a SINGLE action
- [ ] **configure_ methods:** Call multiple `set_` methods, non-essential args default to `None`
- [ ] **No custom __init__:** Unless absolutely required (hardware queries)
- [ ] **SCPI commands:** All write/query strings match the instrument manual
- [ ] **No manual error checking:** Let autocheck or the instrument handle errors
- [ ] **No # comments:** Use docstrings instead
- [ ] **__init__.py updated:** Class is importable
- [ ] **Test notebook created:** `<type>_test.ipynb` covers all methods
- [ ] **Test notebook run:** All tests PASS on real hardware

---

## Quick Reference: Existing Driver Folders

| Folder | Parent Class | Inherits From | Test Notebook |
|---|---|---|---|
| `oscilloscope/` | `Oscilloscope` | `Instrument` | `scope_test.ipynb` |
| `awg/` | `Awg` | `Instrument` | `awg_test.ipynb` |
| `dmm/` | `DMM` | `Instrument` | `dmm_test.ipynb` |
| `lockin/` | `Lockin` | `Instrument` | `lockin_test.ipynb` |
| `sourcemeter/` | `Sourcemeter` | `Instrument` | `sourcemeter_test.ipynb` |
| `pulser/` | `Pulser` | `Instrument` | `pulser_test.ipynb` |
| `dc_callibrator/` | `DCCalibrator` | `Instrument` | `dc_calibrator_test.ipynb` |
| `daq/` | `Daq` | `Instrument` | `daq_test.ipynb` |
| `rf_source/` | `RF_source` | `Instrument` | `rf_source_test.ipynb` |
| `stepper_motor/` | `Stepper` | `Instrument` | `stepper_motor_test.ipynb` |
| `example/` | `Example` | `Instrument` | `example_test.ipynb` (this file) |